# Joining Tables

## Preliminary Note on the Data

Since the data is large (1.5 GB once loaded into a dataframe), we'll define one or more functions per exercise to create our charts.

This way, when each function returns, the temporary variables it used are deleted and the memory can be reused.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas
import seaborn as sns

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s dvf

In [ ]:
df = pandas.read_parquet("dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

In [ ]:
# For readability, keep the 0 in front of single-digit departments
# replacedict = {"code_departement":{x:"0"+x for x in df.code_departement.unique() if len(x) == 1}}
# df.replace(replacedict,inplace=True)

replacedict = {x: "0" + x for x in df.code_departement.unique() if len(x) == 1}
df["code_departement"] = df["code_departement"].cat.rename_categories(replacedict)

In [ ]:
df.info()

## Total Property Value by Region and by Year

In this part, we'll study how to break down a categorical variable by another variable: we'll produce a chart of the total property value exchanged by region, and this value will itself be broken down by year.

To do so, let's start by fetching a dataset that will let us group departments into regions:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

Use this new dataset to join it with our original one.
- *Load `regions.csv` into a `DataFrame`*
- *Add a column containing the region name to your working dataframe, with the [`pandas.DataFrame.join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) function.*
- *Group the data by region then by year with the [`pandas.DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) function.*
- *Use the [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function to display this grouped data. You can refer to the part of the [seaborn tutorial](https://seaborn.pydata.org/tutorial/introduction.html#plots-for-categorical-data) about complex categorical data.*

In [ ]:
def value_by_region_by_year() -> None:
  pass  # Your code here


value_by_region_by_year()

### Solution

In [ ]:
def get_regions_df() -> pandas.DataFrame:
  df = pandas.read_csv(
    "regions.csv",
    index_col="num_dep",
    usecols=["num_dep", "region_name"],
    dtype=dict(region_name="category"),
  )
  return df


get_regions_df()

In [ ]:
def value_by_region_by_year() -> None:
  # join the tables
  joined = df.join(get_regions_df(), on="code_departement")
  # group by region and date, aggregate by summing property values
  data = (
    joined.groupby([joined.region_name, joined.date.dt.year])
    .valeur_fonciere.sum()
    .reset_index()
  )

  # We want to display the results by decreasing total property value of the region
  sort_index = (
    joined.groupby(joined.region_name)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  # display
  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    data, x="region_name", y="valeur_fonciere", hue="date", order=sort_index, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=45)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_region_by_year()

## Sunshine

Using the following two French open data files, [ensoleillement.csv](https://www.data.gouv.fr/fr/datasets/r/4cf82433-9053-4ea8-87d7-c8d409dd7bb7) and [departements.csv](https://www.data.gouv.fr/fr/datasets/r/70cef74f-70b1-495a-8500-c089229c0254),

enrich our original dataset.

In [ ]:
! wget https://www.data.gouv.fr/fr/datasets/r/4cf82433-9053-4ea8-87d7-c8d409dd7bb7 -O ensoleillements.csv
! wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-france.csv -O departements.csv

In [ ]:
def ensoleillement() -> None:
  pass  # Your code here


ensoleillement()

### Solution

In [ ]:
def ensoleillement() -> None:
  sol = pandas.read_csv("ensoleillements.csv")
  dep = pandas.read_csv(
    "departements.csv", usecols=["code_departement", "nom_departement"]
  )
  sol = dep.join(sol)[["code_departement", "Temps d'enseillement (jours/an)"]]
  sol["code_departement"].astype("category")
  return df.merge(sol, on="code_departement", how="left")


ensoleillement()